## PHASE 3: ASSOCIATION RULE MINING (THE MINING PHASE)
**Duration:** 2-3 weeks | **Deliverable:** mined_fraud_patterns.json + fraud_rules_report.xlsx

### ⚠️ FIX: Define Meaningful Items (NOT raw tokens)

The key fix from Gemini's plan: **Don't just throw tokens at Apriori.** Create curated "items" that represent fraud signals.

### Step 3.1: Identify Red Flag Keywords from Fraudulent Postings

**Extract fraudulent-only dataset:**

```python
df_fraud = df[df['fraudulent'] == 1]  # 866 postings
```

**Manually curate red flag patterns** by analyzing fraud postings:

| Red Flag Item | Examples | Frequency in Fraud |
|--------------|----------|-------------------|
| `wire_transfer` | "wire transfer", "wire funds", "send money" | High |
| `upfront_payment` | "upfront", "pay first", "advance payment" | High |
| `unverified_company` | "startup", "new company", "not established" | Moderate |
| `generic_location` | "work from home", "remote", "anywhere" | Moderate |
| `missing_details` | Company logo = 0, no company profile | High |
| `vague_salary` | "salary_range" is empty or very wide ($20k-$200k) | High |
| `urgency_language` | "ASAP", "urgent", "immediate start", "now hiring" | Moderate |
| `bad_grammar` | Multiple spelling errors, poor punctuation | Moderate |
| `too_good_promise` | "easy money", "no experience required", "work from home get rich" | High |
| `data_entry_money` | "data entry" + "commission"/"payment" | High |

### Step 3.2: Create Binary Item Features

**For each red flag, create a binary indicator:**

```python
# Red flag features
df['has_wire_transfer'] = df['consolidated_text'].str.contains(
    r'wire\s+transfer|wire\s+funds|send\s+money', 
    case=False, 
    regex=True
).astype(int)

df['has_upfront_payment'] = df['consolidated_text'].str.contains(
    r'upfront|pay\s+first|advance\s+payment',
    case=False,
    regex=True
).astype(int)

df['has_missing_logo'] = (df['has_company_logo'] == 0).astype(int)

df['has_vague_salary'] = (df['salary_range'].isna() | 
                          (df['salary_range'] == 'Not Specified')).astype(int)

# Repeat for other red flags...
```

**Output:** DataFrame with binary columns like:
```
job_id | has_wire_transfer | has_upfront_payment | has_missing_logo | ... | fraudulent
-------|-------------------|--------------------|--------------------|------|-----------
1      | 0                 | 0                  | 1                  | ... | 0
2      | 1                 | 1                  | 1                  | ... | 1
...
```

### Step 3.3: Apply Apriori Algorithm to Red Flag Combinations

**Now use Apriori on meaningful items:**

```python
from mlxtend.frequent_patterns import apriori, association_rules

# Create itemset from red flag features
itemset_data = df[[
    'has_wire_transfer', 
    'has_upfront_payment', 
    'has_missing_logo',
    'has_vague_salary',
    'has_urgency_language',
    'has_poor_grammar'
]].astype(bool)

# Mine frequent itemsets
frequent_itemsets = apriori(
    itemset_data, 
    min_support=0.05,           # Item appears in 5% of fraudulent postings
    use_colnames=True
)

# Extract association rules
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.6           # 60%+ confidence
)
```

### Step 3.4: Extract & Document Top Rules

**Filter rules for quality:**

```python
# Sort by lift (how much better than random)
rules_sorted = rules.sort_values('lift', ascending=False)

# Keep only high-confidence, high-lift rules
top_rules = rules_sorted[
    (rules_sorted['confidence'] >= 0.60) &
    (rules_sorted['lift'] >= 1.5)
]

print(f"Top 10 Association Rules:\n")
for idx, rule in top_rules.head(10).iterrows():
    antecedent = ', '.join(list(rule['antecedents']))
    consequent = list(rule['consequents'])[0]
    confidence = rule['confidence']
    support = rule['support']
    lift = rule['lift']
    
    print(f"Rule: IF ({antecedent}) THEN fraudulent=1")
    print(f"  Confidence: {confidence:.2%} | Support: {support:.2%} | Lift: {lift:.2f}\n")
```

**Expected Output Example:**
```
Rule: IF (has_wire_transfer AND has_upfront_payment) THEN fraudulent=1
  Confidence: 87% | Support: 3% | Lift: 18.4

Rule: IF (has_missing_logo AND has_vague_salary AND has_urgency_language) THEN fraudulent=1
  Confidence: 74% | Support: 2% | Lift: 15.3

Rule: IF (has_data_entry AND has_payment_promise) THEN fraudulent=1
  Confidence: 81% | Support: 4% | Lift: 16.8
```

### Step 3.5: Compute Pattern Frequency Tables

**Document patterns for the "Discovery" section:**

```python
# Most common red flags in fraudulent postings
fraud_patterns = pd.DataFrame({
    'pattern': [
        'has_wire_transfer',
        'has_upfront_payment',
        'has_missing_logo',
        ...
    ],
    'frequency_in_fraud': [
        df_fraud['has_wire_transfer'].sum(),
        df_fraud['has_upfront_payment'].sum(),
        df_fraud['has_missing_logo'].sum(),
        ...
    ],
    'frequency_in_legit': [
        df_legit['has_wire_transfer'].sum(),
        df_legit['has_upfront_payment'].sum(),
        df_legit['has_missing_logo'].sum(),
        ...
    ]
})

# Compute fraud likelihood ratio
fraud_patterns['fraud_likelihood_ratio'] = (
    fraud_patterns['frequency_in_fraud'] / fraud_patterns['frequency_in_legit']
)

fraud_patterns = fraud_patterns.sort_values('fraud_likelihood_ratio', ascending=False)
print(fraud_patterns.to_string())
```

**Save Results:**
```python
fraud_patterns.to_csv('mined_patterns.csv', index=False)
rules_sorted.to_csv('mined_rules.csv', index=False)
```

**Output Files:**
- `mined_patterns.csv` → Shows which patterns are most discriminative
- `mined_rules.csv` → Association rules with metrics
- `fraud_rules_report.xlsx` → Formatted report with visualizations


In [1]:
import json
from pathlib import Path

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
input_path = Path('../data/processed/cleaned_data.csv')
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(input_path)

if 'consolidated_text' not in df.columns:
    df['consolidated_text'] = (
        df['title'].fillna('') + ' ' +
        df['company_profile'].fillna('') + ' ' +
        df['description'].fillna('') + ' ' +
        df['requirements'].fillna('') + ' ' +
        df['benefits'].fillna('')
    )

print('Loaded shape:', df.shape)
print('Fraud postings:', int((df['fraudulent'] == 1).sum()))

Loaded shape: (17533, 18)
Fraud postings: 846


In [3]:
# Step 3.1: Identify red flag signals in fraudulent postings
df_fraud = df[df['fraudulent'] == 1].copy()
df_legit = df[df['fraudulent'] == 0].copy()

print('Fraud rows:', len(df_fraud))
print('Legit rows:', len(df_legit))

Fraud rows: 846
Legit rows: 16687


In [8]:
import pandas as pd
import re

# Ensure text column is a clean string series
text = df['consolidated_text'].fillna('')

# 1. Define text-pattern flags
text_patterns = {
    'has_wire_transfer': r'wire\s+transfer|wire\s+funds|send\s+money',
    'has_upfront_payment': r'upfront|pay\s+first|advance\s+payment',
    'has_unverified_company': r'\bstartup\b|new\s+company|not\s+established|unknown\s+company',
    'has_generic_location': r'work\s*from\s*home|\bremote\b|\banywhere\b|home\s*based',
    'has_urgency_language': r'\basap\b|urgent|immediate\s+start|now\s+hiring',
    'has_too_good_promise': r'easy\s+money|get\s+rich|quick\s+cash|no\s+experience\s+required',
    'has_data_entry_money': r'data\s+entry.*(commission|payment)|commission.*data\s+entry|paid\s+per\s+entry'
}

for col, pattern in text_patterns.items():
    df[col] = text.str.contains(pattern, case=False, regex=True, na=False).astype(int)

# 2. Metadata-based flags
df['has_missing_logo'] = (df['has_company_logo'].isna() | (df['has_company_logo'] == 0)).astype(int)

vague_values = ['not specified', '', 'nan', 'null']
df['has_vague_salary'] = (
    df['salary_range'].isna() | 
    df['salary_range'].astype(str).str.strip().str.lower().isin(vague_values)
).astype(int)

# 3. Grammar flag: Using re.search with .apply() to bypass pandas engine limitations
# This avoids the 'engine' argument entirely and uses standard Python regex
grammar_pattern = re.compile(r'([!?.,])\1{2,}', re.IGNORECASE)

df['has_poor_grammar'] = (
    text.apply(lambda x: bool(grammar_pattern.search(x))) | 
    (text.str.count(r'[!?.,;:]').fillna(0) > 80)
).astype(int)

# 4. Final Selection
red_flag_cols = list(text_patterns.keys()) + [
    'has_missing_logo', 'has_vague_salary', 'has_poor_grammar'
]

print(df[red_flag_cols + ['fraudulent']].head())

C:\Users\hp\AppData\Local\Temp\ipykernel_11276\2830930642.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[col] = text.str.contains(pattern, case=False, regex=True, na=False).astype(int)


   has_wire_transfer  has_upfront_payment  has_unverified_company  \
0                  0                    0                       1   
1                  0                    0                       1   
2                  0                    0                       0   
3                  0                    0                       0   
4                  0                    0                       0   

   has_generic_location  has_urgency_language  has_too_good_promise  \
0                     0                     0                     0   
1                     1                     0                     0   
2                     0                     0                     0   
3                     0                     0                     0   
4                     0                     0                     0   

   has_data_entry_money  has_missing_logo  has_vague_salary  has_poor_grammar  \
0                     0                 0                 1                 0

In [9]:
# Step 3.3: Apply Apriori on meaningful items
itemset_data = df[red_flag_cols].astype(bool).copy()
itemset_data['is_fraudulent'] = (df['fraudulent'] == 1)

frequent_itemsets = apriori(itemset_data, min_support=0.05, use_colnames=True)

if frequent_itemsets.empty:
    rules = pd.DataFrame()
else:
    rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.60)

print('Frequent itemsets shape:', frequent_itemsets.shape)
print('Rules shape:', rules.shape)

Frequent itemsets shape: (9, 2)
Rules shape: (4, 14)


In [10]:
# Step 3.4: Extract and document top rules (fraud consequent)
if rules.empty:
    print('No rules generated. Try lowering min_support or confidence threshold.')
    rules_sorted = pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction'])
    top_rules = rules_sorted.copy()
    rules_export = rules_sorted.copy()
else:
    rules = rules.copy()
    rules['antecedents_str'] = rules['antecedents'].apply(lambda s: ', '.join(sorted(list(s))))
    rules['consequents_str'] = rules['consequents'].apply(lambda s: ', '.join(sorted(list(s))))

    rules_sorted = rules[rules['consequents'].apply(lambda s: s == frozenset({'is_fraudulent'}))]
    rules_sorted = rules_sorted.sort_values('lift', ascending=False).reset_index(drop=True)

    top_rules = rules_sorted[
        (rules_sorted['confidence'] >= 0.60) &
        (rules_sorted['lift'] >= 1.5)
    ].copy()

    print('Top 10 Association Rules:\n')
    for _, rule in top_rules.head(10).iterrows():
        print(f"Rule: IF ({rule['antecedents_str']}) THEN fraudulent=1")
        print(f"  Confidence: {rule['confidence']:.2%} | Support: {rule['support']:.2%} | Lift: {rule['lift']:.2f}\n")

    rules_export = rules_sorted[[
        'antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 'leverage', 'conviction'
    ]].rename(columns={'antecedents_str': 'antecedents', 'consequents_str': 'consequents'})

Top 10 Association Rules:



In [11]:
# Step 3.5: Compute pattern frequency tables
df_fraud = df[df['fraudulent'] == 1].copy()
df_legit = df[df['fraudulent'] == 0].copy()

fraud_patterns = pd.DataFrame({
    'pattern': red_flag_cols,
    'frequency_in_fraud': [int(df_fraud[c].sum()) for c in red_flag_cols],
    'frequency_in_legit': [int(df_legit[c].sum()) for c in red_flag_cols]
})

# Smoothed ratio to avoid divide-by-zero
fraud_patterns['fraud_likelihood_ratio'] = (
    (fraud_patterns['frequency_in_fraud'] + 1) /
    (fraud_patterns['frequency_in_legit'] + 1)
)

fraud_patterns = fraud_patterns.sort_values('fraud_likelihood_ratio', ascending=False).reset_index(drop=True)
fraud_patterns

,pattern,frequency_in_fraud,frequency_in_legit,fraud_likelihood_ratio
0,has_too_good_promise,17,37,0.473684
1,has_data_entry_money,3,14,0.266667
2,has_missing_logo,567,3037,0.186965
3,has_upfront_payment,3,25,0.153846
4,has_generic_location,165,1319,0.125758
5,has_urgency_language,67,559,0.121429
6,has_poor_grammar,132,2343,0.056741
7,has_vague_salary,625,14062,0.044514
8,has_wire_transfer,0,65,0.015152
9,has_unverified_company,12,2099,0.006190


In [15]:
# Save results to match Phase 3 deliverables
patterns_csv = output_dir / 'mined_patterns.csv'
rules_csv = output_dir / 'mined_rules.csv'
json_path = output_dir / 'mined_fraud_patterns.json'
excel_path = output_dir / 'fraud_rules_report.xlsx'

fraud_patterns.to_csv(patterns_csv, index=False)
rules_export.to_csv(rules_csv, index=False)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'top_rules': top_rules[['antecedents_str', 'support', 'confidence', 'lift']].head(20).to_dict(orient='records'),
        'patterns': fraud_patterns.to_dict(orient='records')
    }, f, indent=2)

with pd.ExcelWriter(excel_path) as writer:
    fraud_patterns.to_excel(writer, sheet_name='fraud_patterns', index=False)
    rules_export.to_excel(writer, sheet_name='association_rules', index=False)
    top_rules[['antecedents_str', 'support', 'confidence', 'lift']].to_excel(writer, sheet_name='top_rules', index=False)

print('Saved:')
print('-', patterns_csv)
print('-', rules_csv)
print('-', json_path)
print('-', excel_path)

Saved:
- ..\data\processed\mined_patterns.csv
- ..\data\processed\mined_rules.csv
- ..\data\processed\mined_fraud_patterns.json
- ..\data\processed\fraud_rules_report.xlsx
